In [ ]:
versioninfo()

# Real ERP Image Processing: Morphology Filter Overview

This notebook plots real ERP images using manually labeled data from Label Studio.
On each run, 3 random **pattern** images and 3 random **no_class** images are selected (each with its own channel + sort variable).

For each filter cell below:
- Row 1: `sort -> z-score -> low-pass -> resize` (reference)
- Row 2: `sort -> z-score -> filter -> low-pass -> resize` (filter + low-pass)
- Row 3: `sort -> z-score -> filter -> resize` (filter only, no low-pass)

In [ ]:
import Pkg
Pkg.activate(".")

using CSV, DataFrames, HDF5, Statistics, Random, Printf
using CairoMakie
using ImageFiltering: imfilter, imgradients, KernelFactors, Kernel, mapwindow
using Images: imresize, dilate, erode, opening, closing, tophat, bothat, morphogradient, morpholaplace
using ImageContrastAdjustment: adjust_histogram, AdaptiveEqualization, Equalization, ContrastStretching

include(joinpath(pwd(), "..", "utils", "erp_image_utils.jl"))
using .ERPImageUtils: gaussian_kernel, zscore_timepoints,
    make_diverging_cmap_zero_centered, clipped_color_stats, clipped_color_stats_filter_row

const SORT_VARIABLES = [:duration, :sac_amplitude, :fix_avgpos_x, :fix_avgpupilsize, :fix_type, :latency]
const SAMPLING_RATE = 512
const PRE_STIM_S = 0.5
const TIME_ZERO_IDX = Int(round(PRE_STIM_S * SAMPLING_RATE)) + 1
const TARGET_SIZE = (64, 64)
const LOWPASS_SIGMA = 75.0f0
const LOWPASS_KERNEL_SIZE = (21, 21)
const FILTER_BORDER = "reflect"

"""
    tv_denoise(img, λ; num_iters=50)

Simple Total Variation denoising via gradient descent (ROF model).
Minimizes: ‖u - img‖² + λ·TV(u), where TV is the isotropic total variation.
"""
function tv_denoise(img::AbstractMatrix{Float32}, λ::Float32; num_iters::Int=50)
    u = copy(img)
    h, w = size(u)
    τ = 0.125f0  # step size (must be < 1/4 for stability)
    for _ in 1:num_iters
        # Compute divergence of normalized gradient
        div = zeros(Float32, h, w)
        for i in 1:h, j in 1:w
            # Forward differences with Neumann boundary
            dx = (i < h ? u[i+1, j] - u[i, j] : 0f0)
            dy = (j < w ? u[i, j+1] - u[i, j] : 0f0)
            grad_norm = sqrt(dx^2 + dy^2 + 1f-8)
            nx = dx / grad_norm
            ny = dy / grad_norm
            # Divergence via backward differences
            div_x = nx - (i > 1 ? (let dx2 = (u[i, j] - u[i-1, j]),
                                        dy2 = (j < w ? u[i-1, j+1] - u[i-1, j] : 0f0),
                                        gn2 = sqrt(dx2^2 + dy2^2 + 1f-8)
                                    dx2 / gn2 end) : 0f0)
            div_y = ny - (j > 1 ? (let dx2 = (i < h ? u[i+1, j-1] - u[i, j-1] : 0f0),
                                        dy2 = (u[i, j] - u[i, j-1]),
                                        gn2 = sqrt(dx2^2 + dy2^2 + 1f-8)
                                    dy2 / gn2 end) : 0f0)
            div[i, j] = div_x + div_y
        end
        @. u = u + τ * (λ * div - (u - img))
    end
    return u
end

function load_erps_from_h5(path::AbstractString)
    return h5open(path, "r") do f
        candidates = ["erps", "/erps", "data", "/data/data_fixations.hdf5", "data/data_fixations.hdf5"]
        for key in candidates
            if haskey(f, key)
                obj = f[key]
                if obj isa HDF5.Dataset
                    return read(obj)
                end
            end
        end

        function first_dataset(g)
            for k in keys(g)
                obj = g[k]
                if obj isa HDF5.Dataset
                    return read(obj)
                elseif obj isa HDF5.Group
                    x = first_dataset(obj)
                    x === nothing || return x
                end
            end
            return nothing
        end

        x = first_dataset(f)
        x === nothing && error("No dataset found in HDF5 file: $path")
        return x
    end
end

fixations_data_dir = joinpath(pwd(), "real_data_sets", "fixations_dataset")
h5_path = joinpath(fixations_data_dir, "data_fixations.hdf5")
events_path = joinpath(fixations_data_dir, "events.csv")

@assert isfile(h5_path) "File not found: $h5_path"
@assert isfile(events_path) "File not found: $events_path"

erps = load_erps_from_h5(h5_path)
events = CSV.read(events_path, DataFrame)

println("Loaded ERP tensor size: ", size(erps), " (channel, time, trial)")
println("Events rows: ", nrow(events))
println("Sort variables: ", SORT_VARIABLES)

# --- Load Label Studio annotations and sample 3 pattern + 3 no_class images ---
const LS_ID_TO_SYMBOL = Dict(0=>:no_class, 1=>:sigmoid, 2=>:one_sided_fan, 3=>:two_sided_fan,
                             4=>:diverging_bar, 5=>:hourglass, 6=>:tilted_bar)

ls_csv_dir = joinpath(pwd(), "results")
ls_required_cols = Set([:annotation_id, :channel, :erp_class, :sort_variable])
ls_dfs = DataFrame[]
skipped_ls_files = String[]

for f in readdir(ls_csv_dir; join=true)
    endswith(f, ".csv") || continue
    df = CSV.read(f, DataFrame)
    cols = Set(Symbol.(names(df)))

    if all(c -> c in cols, ls_required_cols)
        push!(ls_dfs, df)
    else
        push!(skipped_ls_files, basename(f))
    end
end

@assert !isempty(ls_dfs) "No Label-Studio annotation CSV found in $(ls_csv_dir)."

ls_df = vcat(ls_dfs...; cols = :union)

if !isempty(skipped_ls_files)
    println("Skipped non-annotation CSV files in results/: ", join(skipped_ls_files, ", "))
end

required_mask = map(eachrow(ls_df)) do r
    !ismissing(r.erp_class) && !ismissing(r.sort_variable) && !ismissing(r.channel)
end
ls_df = ls_df[required_mask, :]

ls_df.erp_class = [v isa Integer ? Int(v) : parse(Int, string(v)) for v in ls_df.erp_class]
ls_df.channel = [v isa Integer ? Int(v) : parse(Int, string(v)) for v in ls_df.channel]

known_class_mask = map(c -> haskey(LS_ID_TO_SYMBOL, c), ls_df.erp_class)
if any(.!known_class_mask)
    dropped = sort(unique(ls_df.erp_class[.!known_class_mask]))
    println("Dropped rows with unknown erp_class values: ", join(string.(dropped), ", "))
    ls_df = ls_df[known_class_mask, :]
end

ls_df.label = [LS_ID_TO_SYMBOL[c] for c in ls_df.erp_class]
ls_df.sort_var = Symbol.(String.(ls_df.sort_variable))
ls_df.is_pattern = ls_df.erp_class .> 0

pattern_pool = ls_df[ls_df.is_pattern, :]
no_class_pool = ls_df[.!ls_df.is_pattern, :]

println("Label Studio: ", nrow(ls_df), " total images (", nrow(pattern_pool), " pattern, ", nrow(no_class_pool), " no_class)")

# Sample 3 pattern + 3 no_class
selected_pattern = pattern_pool[shuffle(1:nrow(pattern_pool))[1:3], :]
selected_no_class = no_class_pool[shuffle(1:nrow(no_class_pool))[1:3], :]

selected_images = [(channel=r.channel, sort_var=r.sort_var, label=r.label)
                   for r in eachrow(vcat(selected_pattern, selected_no_class))]

println("\nSelected images for this run:")
for (i, s) in enumerate(selected_images)
    println("  [$i] ch$(lpad(string(s.channel), 3, '0')) | $(s.sort_var) | $(s.label)")
end

In [ ]:
function sortvalues_from(df::DataFrame, col::Symbol)
    v = df[!, col]
    if eltype(v) <: Number
        return Float64.(v)
    end
    return collect(v)
end

function preprocess_real_erp_sorted_zscore_image(erps, events::DataFrame, channel::Int, sort_col::Symbol;
    time_zero_idx::Int=TIME_ZERO_IDX)
    @assert 1 <= channel <= size(erps, 1) "channel out of range"
    @assert sort_col in propertynames(events) "sort column not found: $sort_col"

    # 1) Extract post-stimulus matrix (time x trial)
    data = Float32.(erps[channel, time_zero_idx:end, :])
    n = min(size(data, 2), nrow(events))
    data = data[:, 1:n]
    events_n = events[1:n, :]

    # 2) Sort trials by sort variable
    sortvals = sortvalues_from(events_n, sort_col)
    order = sortperm(sortvals)
    data_sorted = data[:, order]

    # 3) Z-score per timepoint over trials
    data_z = zscore_timepoints(data_sorted)

    # 4) Convert to trials x time image
    return Float32.(permutedims(data_z, (2, 1)))
end

function preprocess_real_erp_image(erps, events::DataFrame, channel::Int, sort_col::Symbol;
    time_zero_idx::Int=TIME_ZERO_IDX,
    target_size::Tuple{Int,Int}=TARGET_SIZE,
    low_pass_sigma::Float32=LOWPASS_SIGMA,
    lowpass_kernel_size::Tuple{Int,Int}=LOWPASS_KERNEL_SIZE,
    filter_border::String=FILTER_BORDER)
    img_trials_time = preprocess_real_erp_sorted_zscore_image(erps, events, channel, sort_col;
        time_zero_idx=time_zero_idx)

    # Low-pass filter in original resolution, then resize
    kernel = gaussian_kernel(low_pass_sigma, size(img_trials_time), target_size, lowpass_kernel_size)
    img_lowpass = Float32.(imfilter(img_trials_time, kernel, filter_border))
    return Float32.(imresize(img_lowpass, target_size))
end

function preprocess_real_erp_filter_image(erps, events::DataFrame, channel::Int, sort_col::Symbol;
    filter_fn::Function,
    repeats::Int=1,
    time_zero_idx::Int=TIME_ZERO_IDX,
    target_size::Tuple{Int,Int}=TARGET_SIZE)
    img_trials_time = preprocess_real_erp_sorted_zscore_image(erps, events, channel, sort_col;
        time_zero_idx=time_zero_idx)

    # Filter after z-score, NO low-pass, then resize
    img_filtered = apply_filter_n(img_trials_time, filter_fn; repeats=repeats)
    return Float32.(imresize(img_filtered, target_size))
end

function preprocess_real_erp_filter_lowpass_image(erps, events::DataFrame, channel::Int, sort_col::Symbol;
    filter_fn::Function,
    repeats::Int=1,
    time_zero_idx::Int=TIME_ZERO_IDX,
    target_size::Tuple{Int,Int}=TARGET_SIZE,
    low_pass_sigma::Float32=LOWPASS_SIGMA,
    lowpass_kernel_size::Tuple{Int,Int}=LOWPASS_KERNEL_SIZE,
    filter_border::String=FILTER_BORDER)
    img_trials_time = preprocess_real_erp_sorted_zscore_image(erps, events, channel, sort_col;
        time_zero_idx=time_zero_idx)

    # Filter after z-score, THEN low-pass, then resize
    img_filtered = apply_filter_n(img_trials_time, filter_fn; repeats=repeats)
    kernel = gaussian_kernel(low_pass_sigma, size(img_filtered), target_size, lowpass_kernel_size)
    img_lowpass = Float32.(imfilter(img_filtered, kernel, filter_border))
    return Float32.(imresize(img_lowpass, target_size))
end

function axis_ticks(n::Int)
    mid = Int(cld(n, 2))
    vals = [1, mid, n]
    labels = string.(vals)
    return vals, labels
end

function rebalance_for_diverging_colormap(data::AbstractMatrix; min_side_fraction::Float64=0.08)
    x = Float32.(vec(data))
    n = length(x)
    n == 0 && return Float32.(data), false

    frac_pos = count(>(0f0), x) / n
    frac_neg = count(<(0f0), x) / n

    if min(frac_pos, frac_neg) < min_side_fraction
        return Float32.(data .- median(x)), true
    end

    return Float32.(data), false
end

function apply_filter_n(data::AbstractMatrix, filter_fn::Function; repeats::Int=1)
    out = Float32.(data)
    for _ in 1:repeats
        out = Float32.(filter_fn(out))
    end
    return out
end

function print_pipeline_description(; filter_name::String, repeats::Int,
    time_zero_idx::Int, target_size::Tuple{Int,Int},
    low_pass_sigma::Float32, lowpass_kernel_size::Tuple{Int,Int},
    filter_border::String, clip_quantiles::Tuple{Float64,Float64})
    println()
    println("=== ERP Processing Run: " * filter_name * " ===")
    println("Pipeline steps:")
    println("  Row 1 (reference):")
    println("    sort -> z-score -> low-pass -> resize")
    println("    parameters: time_zero_idx=", time_zero_idx, ", low_pass_sigma=", low_pass_sigma, ", kernel_size=", lowpass_kernel_size, ", border=", filter_border, ", target_size=", target_size)
    println("  Row 2 (filter + low-pass):")
    println("    sort -> z-score -> filter -> low-pass -> resize")
    println("    parameters: filter=", filter_name, ", repeats=", repeats, ", then low_pass_sigma=", low_pass_sigma, ", target_size=", target_size)
    println("  Row 3 (filter only):")
    println("    sort -> z-score -> filter -> resize (no low-pass)")
    println("    parameters: filter=", filter_name, ", repeats=", repeats, ", target_size=", target_size)
    println("  Color clipping:")
    println("    row 1: symmetric around 0 with q_low=", clip_quantiles[1], ", q_high=", clip_quantiles[2])
    println("    rows 2-3: asymmetric q_low=", clip_quantiles[1], ", q_high=", clip_quantiles[2], " (white always at 0)")
end

function plot_real_filter_comparison(erps, events::DataFrame;
    selections::Vector{<:NamedTuple}=selected_images,
    filter_fn::Function,
    filter_name::String,
    repeats::Int=1,
    time_zero_idx::Int=TIME_ZERO_IDX,
    target_size::Tuple{Int,Int}=TARGET_SIZE,
    low_pass_sigma::Float32=LOWPASS_SIGMA,
    lowpass_kernel_size::Tuple{Int,Int}=LOWPASS_KERNEL_SIZE,
    filter_border::String=FILTER_BORDER,
    clip_quantiles::Tuple{Float64,Float64}=(0.01, 0.99))
    n_cols = length(selections)

    print_pipeline_description(;
        filter_name=filter_name,
        repeats=repeats,
        time_zero_idx=time_zero_idx,
        target_size=target_size,
        low_pass_sigma=low_pass_sigma,
        lowpass_kernel_size=lowpass_kernel_size,
        filter_border=filter_border,
        clip_quantiles=clip_quantiles,
    )

    println("Selected images:")
    for (i, s) in enumerate(selections)
        println("  [$i] ch$(lpad(string(s.channel), 3, '0')) | $(s.sort_var) | $(s.label)")
    end

    fig = Figure(size=(420 * n_cols, 1420), figure_padding=24)
    Label(fig[0, 1:(2*n_cols)],
        "Real ERP images | Filter: $filter_name";
        fontsize=22, tellwidth=false)

    selection_rows = NamedTuple[]

    for (j, sel) in enumerate(selections)
        channel = sel.channel
        sort_var = sel.sort_var
        label = sel.label

        # Row 1: reference (low-pass)
        base = preprocess_real_erp_image(erps, events, channel, sort_var;
            time_zero_idx=time_zero_idx,
            target_size=target_size,
            low_pass_sigma=low_pass_sigma,
            lowpass_kernel_size=lowpass_kernel_size,
            filter_border=filter_border)

        # Row 2: filter + low-pass
        filtered_lp = preprocess_real_erp_filter_lowpass_image(erps, events, channel, sort_var;
            time_zero_idx=time_zero_idx,
            target_size=target_size,
            filter_fn=filter_fn,
            repeats=repeats,
            low_pass_sigma=low_pass_sigma,
            lowpass_kernel_size=lowpass_kernel_size,
            filter_border=filter_border)

        # Row 3: filter only (no low-pass)
        filtered = preprocess_real_erp_filter_image(erps, events, channel, sort_var;
            time_zero_idx=time_zero_idx,
            target_size=target_size,
            filter_fn=filter_fn,
            repeats=repeats)

        title_base = string(sort_var) * " | ch" * lpad(string(channel), 3, '0') * " | " * string(label)

        rows_data = [
            (row_i=1, data=base,        suffix="", is_filtered=false),
            (row_i=2, data=filtered_lp, suffix=" | " * filter_name * " + LP", is_filtered=true),
            (row_i=3, data=filtered,    suffix=" | " * filter_name, is_filtered=true),
        ]

        for row in rows_data
            if row.is_filtered
                clipped, crange, tick_vals, tick_labels, cmap = clipped_color_stats_filter_row(row.data;
                    q_low=clip_quantiles[1],
                    q_high=clip_quantiles[2])
            else
                clipped, crange, tick_vals, tick_labels = clipped_color_stats(row.data;
                    q_low=clip_quantiles[1],
                    q_high=clip_quantiles[2])
                cmap = Reverse(:RdBu)
            end

            xtick_vals, xtick_labels = axis_ticks(size(row.data, 2))
            ytick_vals, ytick_labels = axis_ticks(size(row.data, 1))

            img_col = 2 * j - 1
            cb_col = 2 * j

            title_text = title_base * row.suffix

            ax = Axis(fig[row.row_i, img_col];
                title=title_text,
                titlesize=18,
                titlegap=8,
                aspect=DataAspect(),
                xticks=(xtick_vals, xtick_labels),
                yticks=(ytick_vals, ytick_labels),
                xticklabelsize=16,
                yticklabelsize=16,
                xlabel="time",
                ylabel="trials",
                xlabelsize=18,
                ylabelsize=18,
            )

            hm = heatmap!(ax, permutedims(clipped, (2, 1));
                colormap=cmap,
                colorrange=crange)

            Colorbar(fig[row.row_i, cb_col], hm;
                width=18,
                ticklabelsize=15,
                ticks=(tick_vals, tick_labels))

            colsize!(fig.layout, img_col, Fixed(310))
            colsize!(fig.layout, cb_col, Fixed(56))
        end

        push!(selection_rows, (sort_variable=String(sort_var), channel=channel, label=String(label)))
    end

    rowgap!(fig.layout, 16)
    colgap!(fig.layout, 12)
    resize_to_layout!(fig)

    selected_df = DataFrame(selection_rows)
    return fig, selected_df
end

function print_gradient_pipeline_description(; gradient_name::String, time_zero_idx::Int,
    target_size::Tuple{Int,Int}, low_pass_sigma::Float32, lowpass_kernel_size::Tuple{Int,Int},
    filter_border::String, clip_quantiles::Tuple{Float64,Float64},
    rebalance_gradient_rows::Bool, rebalance_min_fraction::Float64)
    println()
    println("=== ERP Gradient Run: " * gradient_name * " ===")
    println("Pipeline steps:")
    println("  Row 1 (reference): sort -> z-score -> low-pass -> resize")
    println("    parameters: time_zero_idx=", time_zero_idx, ", low_pass_sigma=", low_pass_sigma, ", kernel_size=", lowpass_kernel_size, ", border=", filter_border, ", target_size=", target_size)
    println("  Row 2: sort -> z-score -> gradient d/dtime -> low-pass -> resize")
    println("  Row 3: sort -> z-score -> gradient d/dtime -> resize (no low-pass)")
    println("  Row 4: sort -> z-score -> gradient d/dtrial -> low-pass -> resize")
    println("  Row 5: sort -> z-score -> gradient d/dtrial -> resize (no low-pass)")
    println("  Gradient kernel: ", gradient_name, " (computed on full-resolution z-scored image, before resize)")
    println("  Color clipping: q_low=", clip_quantiles[1], ", q_high=", clip_quantiles[2])
    println("  Gradient-row colormap rebalance: ", rebalance_gradient_rows, " (min_side_fraction=", rebalance_min_fraction, ")")
end

function preprocess_real_erp_gradients(erps, events::DataFrame, channel::Int, sort_col::Symbol;
    time_zero_idx::Int=TIME_ZERO_IDX, target_size::Tuple{Int,Int}=TARGET_SIZE,
    low_pass_sigma::Float32=LOWPASS_SIGMA, lowpass_kernel_size::Tuple{Int,Int}=LOWPASS_KERNEL_SIZE,
    filter_border::String=FILTER_BORDER, gradient_kernel=KernelFactors.sobel)
    # Get full-resolution z-scored image
    img_zscore = preprocess_real_erp_sorted_zscore_image(erps, events, channel, sort_col;
        time_zero_idx=time_zero_idx)

    # Row 1: reference with low-pass
    kernel_lp = gaussian_kernel(low_pass_sigma, size(img_zscore), target_size, lowpass_kernel_size)
    base_lp = Float32.(imfilter(img_zscore, kernel_lp, filter_border))
    base_resized = Float32.(imresize(base_lp, target_size))

    # Compute gradients on full resolution BEFORE resize
    g_trial, g_time = imgradients(img_zscore, gradient_kernel, filter_border)

    # With low-pass: gradient -> low-pass -> resize
    g_time_lp = Float32.(imfilter(Float32.(g_time), kernel_lp, filter_border))
    g_trial_lp = Float32.(imfilter(Float32.(g_trial), kernel_lp, filter_border))
    g_time_lp_resized = Float32.(imresize(g_time_lp, target_size))
    g_trial_lp_resized = Float32.(imresize(g_trial_lp, target_size))

    # Without low-pass: gradient -> resize
    g_time_resized = Float32.(imresize(Float32.(g_time), target_size))
    g_trial_resized = Float32.(imresize(Float32.(g_trial), target_size))

    return base_resized, g_time_lp_resized, g_time_resized, g_trial_lp_resized, g_trial_resized
end

function plot_real_gradient_comparison(erps, events::DataFrame;
    selections::Vector{<:NamedTuple}=selected_images,
    gradient_kernel=KernelFactors.sobel, gradient_name::String="Sobel",
    time_zero_idx::Int=TIME_ZERO_IDX, target_size::Tuple{Int,Int}=TARGET_SIZE,
    low_pass_sigma::Float32=LOWPASS_SIGMA, lowpass_kernel_size::Tuple{Int,Int}=LOWPASS_KERNEL_SIZE,
    filter_border::String=FILTER_BORDER, clip_quantiles::Tuple{Float64,Float64}=(0.01, 0.99),
    rebalance_gradient_rows::Bool=true, rebalance_min_fraction::Float64=0.08)
    n_cols = length(selections)

    print_gradient_pipeline_description(; gradient_name=gradient_name,
        time_zero_idx=time_zero_idx, target_size=target_size, low_pass_sigma=low_pass_sigma,
        lowpass_kernel_size=lowpass_kernel_size, filter_border=filter_border,
        clip_quantiles=clip_quantiles, rebalance_gradient_rows=rebalance_gradient_rows,
        rebalance_min_fraction=rebalance_min_fraction)

    println("Selected images:")
    for (i, s) in enumerate(selections)
        println("  [$i] ch$(lpad(string(s.channel), 3, '0')) | $(s.sort_var) | $(s.label)")
    end

    fig = Figure(size=(420 * n_cols, 2300), figure_padding=24)
    Label(fig[0, 1:(2*n_cols)], "Real ERP images | Gradients: " * gradient_name;
        fontsize=22, tellwidth=false)

    selection_rows = NamedTuple[]

    for (j, sel) in enumerate(selections)
        channel = sel.channel
        sort_var = sel.sort_var
        label = sel.label

        base, g_time_lp, g_time_nolp, g_trial_lp, g_trial_nolp = preprocess_real_erp_gradients(
            erps, events, channel, sort_var;
            time_zero_idx=time_zero_idx, target_size=target_size, low_pass_sigma=low_pass_sigma,
            lowpass_kernel_size=lowpass_kernel_size, filter_border=filter_border,
            gradient_kernel=gradient_kernel)

        title_base = string(sort_var) * " | ch" * lpad(string(channel), 3, '0') * " | " * string(label)

        rows = [
            (row_i=1, data=base,          suffix=""),
            (row_i=2, data=g_time_lp,     suffix=" | " * gradient_name * " d/dtime + LP"),
            (row_i=3, data=g_time_nolp,   suffix=" | " * gradient_name * " d/dtime"),
            (row_i=4, data=g_trial_lp,    suffix=" | " * gradient_name * " d/dtrial + LP"),
            (row_i=5, data=g_trial_nolp,  suffix=" | " * gradient_name * " d/dtrial"),
        ]

        for row in rows
            vis_data = row.data
            if row.row_i > 1 && rebalance_gradient_rows
                vis_data, _ = rebalance_for_diverging_colormap(row.data;
                    min_side_fraction=rebalance_min_fraction)
            end

            clipped, crange, tick_vals, tick_labels = clipped_color_stats(vis_data;
                q_low=clip_quantiles[1], q_high=clip_quantiles[2])

            xtick_vals, xtick_labels = axis_ticks(size(row.data, 2))
            ytick_vals, ytick_labels = axis_ticks(size(row.data, 1))

            img_col = 2 * j - 1
            cb_col = 2 * j

            title_text = title_base * row.suffix

            ax = Axis(fig[row.row_i, img_col];
                title=title_text, titlesize=18, titlegap=8, aspect=DataAspect(),
                xticks=(xtick_vals, xtick_labels), yticks=(ytick_vals, ytick_labels),
                xticklabelsize=16, yticklabelsize=16,
                xlabel="time", ylabel="trials", xlabelsize=18, ylabelsize=18)

            hm = heatmap!(ax, permutedims(clipped, (2, 1));
                colormap=Reverse(:RdBu), colorrange=crange)

            Colorbar(fig[row.row_i, cb_col], hm;
                width=18, ticklabelsize=15, ticks=(tick_vals, tick_labels))

            colsize!(fig.layout, img_col, Fixed(310))
            colsize!(fig.layout, cb_col, Fixed(56))
        end

        push!(selection_rows, (sort_variable=String(sort_var), channel=channel, label=String(label)))
    end

    rowgap!(fig.layout, 16)
    colgap!(fig.layout, 12)
    resize_to_layout!(fig)

    selected_df = DataFrame(selection_rows)
    return fig, selected_df
end

In [ ]:
# Filter 1/8: Erosion
fig_erode, channels_erode = plot_real_filter_comparison(erps, events;
    filter_fn=erode,
    filter_name="Erosion")
display(fig_erode)
channels_erode


In [ ]:
# Filter 2/8: Dilation
fig_dilate, channels_dilate = plot_real_filter_comparison(erps, events;
    filter_fn=dilate,
    filter_name="Dilation")
display(fig_dilate)
channels_dilate


In [ ]:
# Filter 3/8: Opening
fig_opening, channels_opening = plot_real_filter_comparison(erps, events;
    filter_fn=opening,
    filter_name="Opening")
display(fig_opening)
channels_opening


In [ ]:
# Filter 4/8: Closing
fig_closing, channels_closing = plot_real_filter_comparison(erps, events;
    filter_fn=closing,
    filter_name="Closing")
display(fig_closing)
channels_closing


In [ ]:
# Filter 5/8: Tophat
fig_tophat, channels_tophat = plot_real_filter_comparison(erps, events;
    filter_fn=tophat,
    filter_name="Tophat")
display(fig_tophat)
channels_tophat


In [ ]:
# Filter 6/8: Bothat (Bottom Hat)
fig_bothat, channels_bothat = plot_real_filter_comparison(erps, events;
    filter_fn=bothat,
    filter_name="Bothat")
display(fig_bothat)
channels_bothat


In [ ]:
# Filter 7/8: Morphological Gradient
fig_morphogradient, channels_morphogradient = plot_real_filter_comparison(erps, events;
    filter_fn=morphogradient,
    filter_name="Morphological Gradient")
display(fig_morphogradient)
channels_morphogradient


In [ ]:
# Filter 8/8: Morphological Laplace
fig_morpholaplace, channels_morpholaplace = plot_real_filter_comparison(erps, events;
    filter_fn=morpholaplace,
    filter_name="Morphological Laplace")
display(fig_morpholaplace)
channels_morpholaplace


## Kernel-based Filters from ImageFiltering.jl

This section applies convolution kernels from `ImageFiltering.Kernel` and `ImageFiltering.KernelFactors`.
For edge-detection kernels (Sobel, Prewitt, Scharr, Bickley, Ando3/4/5), the gradient magnitude
`√(g_x² + g_y²)` is computed via `imgradients`.
For smoothing/detection kernels (Gaussian, DoG, LoG, Laplacian, Gabor, Moffat), `imfilter` is used directly.

Pipeline per cell:
- Row 1: `sort → z-score → low-pass → resize` (reference)
- Row 2: `sort → z-score → kernel filter → resize`

In [ ]:
# Kernel Filter 9: Sobel (Gradient Magnitude)
fig_sobel, channels_sobel = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        g1, g2 = imgradients(Float32.(data), KernelFactors.sobel, "reflect")
        Float32.(sqrt.(Float32.(g1).^2 .+ Float32.(g2).^2))
    end,
    filter_name="Sobel Gradient Magnitude")
display(fig_sobel)
channels_sobel

In [ ]:
# Kernel Filter 10: Prewitt (Gradient Magnitude)
fig_prewitt, channels_prewitt = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        g1, g2 = imgradients(Float32.(data), KernelFactors.prewitt, "reflect")
        Float32.(sqrt.(Float32.(g1).^2 .+ Float32.(g2).^2))
    end,
    filter_name="Prewitt Gradient Magnitude")
display(fig_prewitt)
channels_prewitt

In [ ]:
# Kernel Filter 11: Scharr (Gradient Magnitude)
fig_scharr, channels_scharr = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        g1, g2 = imgradients(Float32.(data), KernelFactors.scharr, "reflect")
        Float32.(sqrt.(Float32.(g1).^2 .+ Float32.(g2).^2))
    end,
    filter_name="Scharr Gradient Magnitude")
display(fig_scharr)
channels_scharr

In [ ]:
# Kernel Filter 12: Bickley (Gradient Magnitude)
fig_bickley, channels_bickley = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        g1, g2 = imgradients(Float32.(data), KernelFactors.bickley, "reflect")
        Float32.(sqrt.(Float32.(g1).^2 .+ Float32.(g2).^2))
    end,
    filter_name="Bickley Gradient Magnitude")
display(fig_bickley)
channels_bickley

In [ ]:
# Kernel Filter 13: Ando3 (Gradient Magnitude)
fig_ando3, channels_ando3 = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        g1, g2 = imgradients(Float32.(data), KernelFactors.ando3, "reflect")
        Float32.(sqrt.(Float32.(g1).^2 .+ Float32.(g2).^2))
    end,
    filter_name="Ando3 Gradient Magnitude")
display(fig_ando3)
channels_ando3

In [ ]:
# Kernel Filter 14: Ando4 (Gradient Magnitude)
fig_ando4, channels_ando4 = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        g1, g2 = imgradients(Float32.(data), KernelFactors.ando4, "reflect")
        Float32.(sqrt.(Float32.(g1).^2 .+ Float32.(g2).^2))
    end,
    filter_name="Ando4 Gradient Magnitude")
display(fig_ando4)
channels_ando4

In [ ]:
# Kernel Filter 15: Ando5 (Gradient Magnitude)
fig_ando5, channels_ando5 = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        g1, g2 = imgradients(Float32.(data), KernelFactors.ando5, "reflect")
        Float32.(sqrt.(Float32.(g1).^2 .+ Float32.(g2).^2))
    end,
    filter_name="Ando5 Gradient Magnitude")
display(fig_ando5)
channels_ando5

In [ ]:
# Kernel Filter 16: Gaussian (σ=1.0)
fig_gauss, channels_gauss = plot_real_filter_comparison(erps, events;
    filter_fn=data -> Float32.(imfilter(Float32.(data), Kernel.gaussian((1.0, 1.0)), "reflect")),
    filter_name="Gaussian (σ=1.0)")
display(fig_gauss)
channels_gauss

In [ ]:
# Kernel Filter 17: Difference of Gaussians (DoG)
fig_dog, channels_dog = plot_real_filter_comparison(erps, events;
    filter_fn=data -> Float32.(imfilter(Float32.(data), Kernel.DoG((1.0, 1.0)), "reflect")),
    filter_name="DoG (σ=1.0)")
display(fig_dog)
channels_dog

In [ ]:
# Kernel Filter 18: Laplacian of Gaussian (LoG)
fig_log, channels_log = plot_real_filter_comparison(erps, events;
    filter_fn=data -> Float32.(imfilter(Float32.(data), Kernel.LoG((1.0, 1.0)), "reflect")),
    filter_name="LoG (σ=1.0)")
display(fig_log)
channels_log

In [ ]:
# Kernel Filter 19: Laplacian
fig_laplacian, channels_laplacian = plot_real_filter_comparison(erps, events;
    filter_fn=data -> Float32.(imfilter(Float32.(data), Kernel.Laplacian(), "reflect")),
    filter_name="Laplacian")
display(fig_laplacian)
channels_laplacian

In [ ]:
# Kernel Filter 20: Gabor (θ=0, σ=3, λ=10, γ=0.5, ψ=0)
fig_gabor, channels_gabor = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        kr, ki = Kernel.gabor(21, 21, 3.0, 0.0, 10.0, 0.5, 0.0)
        Float32.(sqrt.(imfilter(Float32.(data), kr, "reflect").^2 .+ imfilter(Float32.(data), ki, "reflect").^2))
    end,
    filter_name="Gabor (θ=0, σ=3, λ=10)")
display(fig_gabor)
channels_gabor

In [ ]:
# Kernel Filter 21: Moffat (α=2.0, β=2.5)
fig_moffat, channels_moffat = plot_real_filter_comparison(erps, events;
    filter_fn=data -> Float32.(imfilter(Float32.(data), Kernel.moffat(2.0, 2.5), "reflect")),
    filter_name="Moffat (α=2.0, β=2.5)")
display(fig_moffat)
channels_moffat

## Advanced Preprocessing Techniques

This section applies additional image processing techniques aimed at enhancing ERP patterns for CNN classification:
- **Denoising**: Median Filter, Total Variation (ROF) Denoising
- **Contrast Enhancement**: CLAHE (Adaptive Histogram Equalization), Histogram Equalization, Contrast Stretching
- **Edge & Structure Enhancement**: Unsharp Masking, Gabor Filter Bank (multiple orientations)
- **Multi-scale Analysis**: Multi-scale DoG, Gaussian Pyramid

Pipeline per cell:
- Row 1: `sort → z-score → low-pass → resize` (reference)
- Row 2: `sort → z-score → technique → resize`

In [ ]:
# Filter 22: Median Filter (3×3)
# Non-linear denoising that preserves edges while removing salt-and-pepper noise.
fig_median3, channels_median3 = plot_real_filter_comparison(erps, events;
    filter_fn=data -> Float32.(mapwindow(median!, Float32.(data), (3, 3))),
    filter_name="Median Filter (3×3)")
display(fig_median3)
channels_median3

In [ ]:
# Filter 23: Median Filter (5×5)
# Larger median window for stronger denoising.
fig_median5, channels_median5 = plot_real_filter_comparison(erps, events;
    filter_fn=data -> Float32.(mapwindow(median!, Float32.(data), (5, 5))),
    filter_name="Median Filter (5×5)")
display(fig_median5)
channels_median5

In [ ]:
# Filter 24: Total Variation (ROF) Denoising (λ=0.1)
# Rudin-Osher-Fatemi denoising via gradient descent. Removes noise while preserving edges.
# λ controls regularization strength: larger = smoother.
fig_rof01, channels_rof01 = plot_real_filter_comparison(erps, events;
    filter_fn=data -> tv_denoise(Float32.(data), 0.1f0; num_iters=50),
    filter_name="TV/ROF Denoising (λ=0.1)")
display(fig_rof01)
channels_rof01

In [ ]:
# Filter 25: Total Variation (ROF) Denoising (λ=0.5)
# Stronger TV regularization for more aggressive smoothing.
fig_rof05, channels_rof05 = plot_real_filter_comparison(erps, events;
    filter_fn=data -> tv_denoise(Float32.(data), 0.5f0; num_iters=50),
    filter_name="TV/ROF Denoising (λ=0.5)")
display(fig_rof05)
channels_rof05

In [ ]:
# Filter 26: Unsharp Masking (σ=2.0, amount=1.5)
# Sharpens the image by subtracting a blurred version: result = original + amount*(original - blurred)
# Enhances fine details and edges in ERP patterns.
fig_unsharp, channels_unsharp = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        d = Float32.(data)
        blurred = Float32.(imfilter(d, Kernel.gaussian((2.0, 2.0)), "reflect"))
        Float32.(d .+ 1.5f0 .* (d .- blurred))
    end,
    filter_name="Unsharp Masking (σ=2.0, α=1.5)")
display(fig_unsharp)
channels_unsharp

In [ ]:
# Filter 27: Unsharp Masking (σ=1.0, amount=2.0)
# Stronger sharpening with smaller blur radius.
fig_unsharp2, channels_unsharp2 = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        d = Float32.(data)
        blurred = Float32.(imfilter(d, Kernel.gaussian((1.0, 1.0)), "reflect"))
        Float32.(d .+ 2.0f0 .* (d .- blurred))
    end,
    filter_name="Unsharp Masking (σ=1.0, α=2.0)")
display(fig_unsharp2)
channels_unsharp2

In [ ]:
# Filter 28: CLAHE (Adaptive Histogram Equalization)
# Contrast Limited Adaptive Histogram Equalization enhances local contrast.
# Data is normalized to [0,1] for adjust_histogram, then mapped back to original range.
fig_clahe, channels_clahe = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        d = Float32.(data)
        lo, hi = extrema(d)
        rng = hi - lo
        rng = rng == 0f0 ? 1f0 : rng
        d_norm = (d .- lo) ./ rng
        d_eq = Float32.(adjust_histogram(d_norm, AdaptiveEqualization(nbins=256, rblocks=8, cblocks=8, clip=0.1)))
        Float32.(d_eq .* rng .+ lo)
    end,
    filter_name="CLAHE (clip=0.1)")
display(fig_clahe)
channels_clahe

In [ ]:
# Filter 29: Histogram Equalization
# Global histogram equalization spreads intensity values uniformly across the full range.
fig_histeq, channels_histeq = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        d = Float32.(data)
        lo, hi = extrema(d)
        rng = hi - lo
        rng = rng == 0f0 ? 1f0 : rng
        d_norm = (d .- lo) ./ rng
        d_eq = Float32.(adjust_histogram(d_norm, Equalization(nbins=256)))
        Float32.(d_eq .* rng .+ lo)
    end,
    filter_name="Histogram Equalization")
display(fig_histeq)
channels_histeq

In [ ]:
# Filter 30: Contrast Stretching (t=0.5, slope=5.0)
# Sigmoidal contrast stretching: f(x) = 1/(1 + (t/x)^slope)
# Enhances contrast by stretching mid-range intensities.
fig_cstretch, channels_cstretch = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        d = Float32.(data)
        lo, hi = extrema(d)
        rng = hi - lo
        rng = rng == 0f0 ? 1f0 : rng
        d_norm = (d .- lo) ./ rng
        d_eq = Float32.(adjust_histogram(d_norm, ContrastStretching(t=0.5, slope=5.0)))
        Float32.(d_eq .* rng .+ lo)
    end,
    filter_name="Contrast Stretching (t=0.5, slope=5)")
display(fig_cstretch)
channels_cstretch

In [ ]:
# Filter 31: Multi-scale DoG (σ₁=0.5, σ₂=2.0, σ₃=4.0)
# Combines Difference of Gaussians at 3 scales into a single feature map.
# Captures edges and blobs at different spatial frequencies.
fig_msdog, channels_msdog = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        d = Float32.(data)
        dog1 = Float32.(imfilter(d, Kernel.DoG((0.5, 0.5)), "reflect"))
        dog2 = Float32.(imfilter(d, Kernel.DoG((2.0, 2.0)), "reflect"))
        dog3 = Float32.(imfilter(d, Kernel.DoG((4.0, 4.0)), "reflect"))
        Float32.(sqrt.(dog1.^2 .+ dog2.^2 .+ dog3.^2))
    end,
    filter_name="Multi-scale DoG (σ=0.5,2,4)")
display(fig_msdog)
channels_msdog

In [ ]:
# Filter 32: DoG (σ=0.5) — fine scale
fig_dog_fine, channels_dog_fine = plot_real_filter_comparison(erps, events;
    filter_fn=data -> Float32.(imfilter(Float32.(data), Kernel.DoG((0.5, 0.5)), "reflect")),
    filter_name="DoG (σ=0.5, fine)")
display(fig_dog_fine)
channels_dog_fine

In [ ]:
# Filter 33: DoG (σ=4.0) — coarse scale
fig_dog_coarse, channels_dog_coarse = plot_real_filter_comparison(erps, events;
    filter_fn=data -> Float32.(imfilter(Float32.(data), Kernel.DoG((4.0, 4.0)), "reflect")),
    filter_name="DoG (σ=4.0, coarse)")
display(fig_dog_coarse)
channels_dog_coarse

In [ ]:
# Filter 34: Gabor Filter Bank (4 orientations: 0°, 45°, 90°, 135°)
# Combines Gabor responses at multiple orientations into a single energy map.
# Captures oriented structures (bars, fans, tilted patterns) at all angles.
fig_gabor_bank, channels_gabor_bank = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        d = Float32.(data)
        energy = zeros(Float32, size(d))
        for θ in [0.0, π/4, π/2, 3π/4]
            kr, ki = Kernel.gabor(21, 21, 3.0, θ, 10.0, 0.5, 0.0)
            resp_r = Float32.(imfilter(d, kr, "reflect"))
            resp_i = Float32.(imfilter(d, ki, "reflect"))
            energy .+= resp_r.^2 .+ resp_i.^2
        end
        Float32.(sqrt.(energy))
    end,
    filter_name="Gabor Bank (4 orient., σ=3, λ=10)")
display(fig_gabor_bank)
channels_gabor_bank

In [ ]:
# Filter 35: Gabor (θ=π/2, σ=3, λ=10) — vertical orientation
# Single Gabor at 90° to detect vertical patterns (e.g., bars).
fig_gabor90, channels_gabor90 = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        kr, ki = Kernel.gabor(21, 21, 3.0, π/2, 10.0, 0.5, 0.0)
        Float32.(sqrt.(imfilter(Float32.(data), kr, "reflect").^2 .+ imfilter(Float32.(data), ki, "reflect").^2))
    end,
    filter_name="Gabor (θ=90°, σ=3, λ=10)")
display(fig_gabor90)
channels_gabor90

In [ ]:
# Filter 36: Gaussian Pyramid Residual (level 2)
# Computes residual between original and 2x downsampled+upsampled version.
# Captures mid-frequency details lost at coarser resolution.
fig_pyrres, channels_pyrres = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        d = Float32.(data)
        h, w = size(d)
        # Blur, downsample to half, then upsample back
        blurred = Float32.(imfilter(d, Kernel.gaussian((2.0, 2.0)), "reflect"))
        down = Float32.(imresize(blurred, (div(h, 2), div(w, 2))))
        up = Float32.(imresize(down, (h, w)))
        Float32.(d .- up)
    end,
    filter_name="Gaussian Pyramid Residual (L2)")
display(fig_pyrres)
channels_pyrres

In [ ]:
# Filter 37: Gaussian Pyramid Residual (level 3)
# Deeper pyramid: residual between original and 4x downsampled+upsampled version.
# Captures lower-frequency details.
fig_pyrres3, channels_pyrres3 = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        d = Float32.(data)
        h, w = size(d)
        blurred = Float32.(imfilter(d, Kernel.gaussian((4.0, 4.0)), "reflect"))
        down = Float32.(imresize(blurred, (div(h, 4), div(w, 4))))
        up = Float32.(imresize(down, (h, w)))
        Float32.(d .- up)
    end,
    filter_name="Gaussian Pyramid Residual (L3)")
display(fig_pyrres3)
channels_pyrres3

In [ ]:
# Filter 38: Median + Unsharp (denoise then sharpen)
# Two-step pipeline: first remove noise with median filter, then sharpen edges.
fig_med_unsharp, channels_med_unsharp = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        d = Float32.(data)
        # Step 1: denoise with median
        denoised = Float32.(mapwindow(median!, d, (3, 3)))
        # Step 2: unsharp masking
        blurred = Float32.(imfilter(denoised, Kernel.gaussian((1.5, 1.5)), "reflect"))
        Float32.(denoised .+ 1.5f0 .* (denoised .- blurred))
    end,
    filter_name="Median(3) + Unsharp(σ=1.5, α=1.5)")
display(fig_med_unsharp)
channels_med_unsharp

In [ ]:
# Filter 39: TV/ROF Denoise + Sobel Edge (denoise then detect edges)
# Two-step pipeline: TV denoising to clean up, then Sobel gradient magnitude.
fig_rof_sobel, channels_rof_sobel = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        denoised = tv_denoise(Float32.(data), 0.2f0; num_iters=50)
        g1, g2 = imgradients(denoised, KernelFactors.sobel, "reflect")
        Float32.(sqrt.(Float32.(g1).^2 .+ Float32.(g2).^2))
    end,
    filter_name="ROF(λ=0.2) + Sobel Magnitude")
display(fig_rof_sobel)
channels_rof_sobel

In [ ]:
# Filter 40: CLAHE + Sobel (enhance contrast then detect edges)
# Two-step: CLAHE for local contrast enhancement, then Sobel edge detection.
fig_clahe_sobel, channels_clahe_sobel = plot_real_filter_comparison(erps, events;
    filter_fn=data -> begin
        d = Float32.(data)
        lo, hi = extrema(d)
        rng = hi - lo
        rng = rng == 0f0 ? 1f0 : rng
        d_norm = (d .- lo) ./ rng
        d_eq = Float32.(adjust_histogram(d_norm, AdaptiveEqualization(nbins=256, rblocks=8, cblocks=8, clip=0.1)))
        d_eq = Float32.(d_eq .* rng .+ lo)
        g1, g2 = imgradients(d_eq, KernelFactors.sobel, "reflect")
        Float32.(sqrt.(Float32.(g1).^2 .+ Float32.(g2).^2))
    end,
    filter_name="CLAHE + Sobel Magnitude")
display(fig_clahe_sobel)
channels_clahe_sobel

## Gradient Rows (d/dtime, d/dtrial)

This section uses the same selected labeled images (3 pattern + 3 no_class) and shows five rows per image:
1) reference ERP (`sort -> z-score -> low-pass -> resize`)
2) time gradient with low-pass (`sort -> z-score -> gradient d/dtime -> low-pass -> resize`)
3) time gradient without low-pass (`sort -> z-score -> gradient d/dtime -> resize`)
4) trial gradient with low-pass (`sort -> z-score -> gradient d/dtrial -> low-pass -> resize`)
5) trial gradient without low-pass (`sort -> z-score -> gradient d/dtrial -> resize`)

Gradients are computed on the **full-resolution** z-scored image (before resize), using `ImageFiltering.imgradients` with `KernelFactors.sobel`.

In [ ]:
# Gradient comparison: Sobel (library gradients)
fig_gradients_sobel, channels_gradients_sobel = plot_real_gradient_comparison(erps, events;
    gradient_kernel=KernelFactors.sobel,
    gradient_name="Sobel")
display(fig_gradients_sobel)
channels_gradients_sobel